In [1]:
import pandas as pd

calendar = pd.read_csv('Data/calendar.csv')
items = pd.read_csv('Data/items_per_season.csv')
cost_amount = pd.read_csv('Data/cost_amount.csv')
mail_hashed_all = pd.read_csv('Data/mail_hashed_all.csv')
customers_fy_16_17 = pd.read_csv('Data/customers_fy_16_17.csv')
customers_fy_23_24 = pd.read_csv('Data/customers_fy_23_24.csv')
items_2016 = pd.read_csv('Data/items_2016.csv')
items_2022 = pd.read_csv('Data/items_2022.csv')
orders_fy_16_17 = pd.read_csv('Data/order_invoice_refunds_fy_16_17.csv')
orders_fy_22_23 = pd.read_csv('Data/order_invoice_refunds_fy_22_23.csv')

In [2]:
# Kunden zusammenführen & H_ID anlegen
customers_base = pd.concat([customers_fy_16_17, customers_fy_23_24], ignore_index=True)
customers_base = customers_base.drop_duplicates().reset_index(drop=True)

customers_all_hid = customers_base.merge(mail_hashed_all, on='CUSTOMER_NO', how='left')
customers_all_hid = customers_all_hid.rename(columns={'EMAIL_HASHED': 'H_ID'})

print(f"Kunden gesamt (dedup): {len(customers_base):,}")
print(f"Kunden mit H_ID:       {customers_all_hid['H_ID'].notna().sum():,}")
print(f"Unique H_IDs:          {customers_all_hid['H_ID'].nunique():,}")
display(customers_all_hid.head())


Kunden gesamt (dedup): 592,547
Kunden mit H_ID:       592,402
Unique H_IDs:          534,795


,CUSTOMER_NO,CUSTOMER_TYPE,COUNTRY_REGION_CODE,POST_CODE,CITY,IS_BLOCKED,IS_ACTIVE_IN_L2Y,IS_ACTIVE_LAST_FY,LAST_ORDER_DATE,CUSTOMER_ORDER_TYPE,...,FIRST_ORDER_NET_AMOUNT_AFTER_RETURNS,FIRST_ORDER_GROSS_AMOUNT_BEFORE_RETURNS,FIRST_ORDER_GROSS_AMOUNT_AFTER_RETURNS,IS_SUBSCRIBED_FOR,IS_OS_B2B_CUSTOMER,DAYS_SINCE_FIRST_ORDER,DAYS_SINCE_LAST_ORDER,CUSTOMER_CREATED_AT,HAS_CUSTOMER_ACCOUNT,H_ID
0,D400229,privat,DE,30926,Seelze,False,False,not active last fy,2021-04-21,Repeat Customer,...,50.08,59.60,59.60,NaN,False,2326,1820,2019-12-02,False,8a2ada2273d2d041a280657485c9bced5d5a6045
1,D599141,privat,AT,2345,Brunn am Gebirge,False,False,not active last fy,2021-11-13,Repeat Customer,...,133.17,158.47,158.47,AFFENZAHN,False,2036,1614,2020-09-17,False,755baadc15b3ca05d0efa37950172c44fbc183d1
2,D609579,privat,DE,56179,Vallendar,False,False,not active last fy,2021-11-28,Serial Customer,...,86.12,102.48,102.48,NaN,False,2017,1599,2020-10-06,False,598ccdeee59d786cd8da19541fc30b384aff097c
3,D385993,privat,FR,88640,Granges sur vologne,False,False,not active last fy,2019-11-21,One-time Customer,...,83.25,99.07,99.07,AFFENZAHN,False,2337,2337,2019-11-21,False,13ff974b444f85193369e43719b901ab71f32eeb
4,D931939,privat,DE,06729,Elsteraue,False,True,active last fy,2025-09-13,Serial Customer,...,0.00,42.00,0.00,NaN,False,1655,214,2021-10-03,False,8ace8f5270016168af0b1805bec411092ef13a76


In [3]:
#Mapping + Kundenprofil pro H_ID
cust_hid_map = (
    customers_all_hid[['CUSTOMER_NO', 'H_ID']]
    .drop_duplicates(subset='CUSTOMER_NO')
)

customers_hid_profile = (
    customers_all_hid
    .drop_duplicates(subset='H_ID')
    .reset_index(drop=True)
)

print(f"Mapping-Einträge (CUSTOMER_NO → H_ID): {len(cust_hid_map):,}")
print(f"Kundenprofile pro H_ID:                {len(customers_hid_profile):,}")
print(f"Spalten im Kundenprofil:               {list(customers_hid_profile.columns)}")


Mapping-Einträge (CUSTOMER_NO → H_ID): 535,583
Kundenprofile pro H_ID:                534,796
Spalten im Kundenprofil:               ['CUSTOMER_NO', 'CUSTOMER_TYPE', 'COUNTRY_REGION_CODE', 'POST_CODE', 'CITY', 'IS_BLOCKED', 'IS_ACTIVE_IN_L2Y', 'IS_ACTIVE_LAST_FY', 'LAST_ORDER_DATE', 'CUSTOMER_ORDER_TYPE', 'CUSTOMER_ORDER_TYPE_ADJ', 'CUSTOMER_RETURN_TYPE', 'RETURN_COUNT_PARTIAL_OR_COMPLETE', 'RETURN_COUNT_COMPLETE', 'ORDER_COUNT', 'ORDER_COUNT_AFTER_COMPLETE_RETURNS', 'REKLAMATION_COUNT', 'FIRST_ORDER_DATE', 'SECOND_ORDER_DATE', 'DAYS_FIRST_TO_SECOND_ORDER', 'DAYS_BETWEEN_FIRST_AND_LAST_ORDER', 'AVG_DAYS_BETWEEN_ORDERS', 'CLV', 'TOTAL_ORDER_VALUE', 'TOTAL_RETURN_VALUE', 'TOTAL_ORDER_VALUE_BEFORE_RETURNS', 'DISCOUNT_SHARE_PER_CUSTOMER', 'AVERAGE_ORDER_VALUE', 'AVERAGE_ORDER_VALUE_BEFORE_RETURNS', 'FIRST_ORDER_NET_AMOUNT_BEFORE_RETURNS', 'FIRST_ORDER_NET_AMOUNT_AFTER_RETURNS', 'FIRST_ORDER_GROSS_AMOUNT_BEFORE_RETURNS', 'FIRST_ORDER_GROSS_AMOUNT_AFTER_RETURNS', 'IS_SUBSCRIBED_FOR', 'IS_OS_

In [4]:
#Items kombinieren
items_all_cols = (
    pd.concat([items_2016, items_2022], ignore_index=True)
    .drop_duplicates(subset='ITEM_NO')
    .reset_index(drop=True)
)

print(f"Items gesamt (dedup): {len(items_all_cols):,}")
print(f"Spalten:              {list(items_all_cols.columns)}")
display(items_all_cols.head())


Items gesamt (dedup): 5,257
Spalten:              ['ITEM_NO', 'ITEM_NO_PREDECESSOR_1', 'ITEM_NO_PREDECESSOR_2', 'BRAND', 'MAIN_CATEGORY', 'SUB_CATEGORY', 'PRODUCT_TYPE', 'MATERIAL', 'SIZE', 'CATEGORY_KEY', 'PLANNING_CATEGORY_KEY', 'APPEARANCE', 'APPEARANCE_EN', 'CURRENT_PRODUCT_STATUS']


,ITEM_NO,ITEM_NO_PREDECESSOR_1,ITEM_NO_PREDECESSOR_2,BRAND,MAIN_CATEGORY,SUB_CATEGORY,PRODUCT_TYPE,MATERIAL,SIZE,CATEGORY_KEY,PLANNING_CATEGORY_KEY,APPEARANCE,APPEARANCE_EN,CURRENT_PRODUCT_STATUS
0,AFZ-SKN-128-20083,NaN,NaN,AFFENZAHN,FOOTWEAR,LOWCUTS,HAPPY,POLYESTER KNIT,28,AFFENZAHN~FOOTWEAR~LOWCUTS~HAPPY,AFFENZAHN~FOOTWEAR~LOWCUTS~HAPPY~~,DRAGON,NaN,old - older than previous season
1,AFZ-SES-129-001,NaN,NaN,AFFENZAHN,FOOTWEAR ACC,INSERT SOLES,INSERT SOLES,POLYESTER,29,AFFENZAHN~FOOTWEAR ACC~INSERT SOLES~INSERT SOLES,AFFENZAHN~FOOTWEAR ACC~INSERT SOLES~INSERT SOL...,TIGER,NaN,old - older than previous season
2,AFZ-FAS-004-001,NaN,NaN,AFFENZAHN,BAGS,BACKPACKS,SMALL FRIEND,POLYESTER CANVAS,onesize,AFFENZAHN~BAGS~BACKPACKS~SMALL FRIEND,AFFENZAHN~BAGS~BACKPACKS~SMALL FRIEND~~,Tiger,Tiger,old - previous season
3,AFZ-FAS-002-002,NaN,NaN,AFFENZAHN,BAGS,BACKPACKS,SMALL FRIEND,POLYESTER,onesize,AFFENZAHN~BAGS~BACKPACKS~SMALL FRIEND,AFFENZAHN~BAGS~BACKPACKS~SMALL FRIEND~~,Löwe,Lion,old - older than previous season
4,AFZ-SLB-128-104,NaN,NaN,AFFENZAHN,FOOTWEAR,LOWBOOTS,LOWBOOTS,LEATHER,28,AFFENZAHN~FOOTWEAR~LOWBOOTS~LOWBOOTS,AFFENZAHN~FOOTWEAR~LOWBOOTS~LOWBOOTS~~,LION,NaN,NaN


In [5]:
#Orders kombinieren, H_ID + Items + Kosten joinen
orders_base = pd.concat([orders_fy_16_17, orders_fy_22_23], ignore_index=True)
orders_base = orders_base.drop_duplicates().reset_index(drop=True)
orders_base['ORDER_DATE'] = pd.to_datetime(orders_base['ORDER_DATE'], errors='coerce')

orders_enriched_hid = (
    orders_base
    .merge(cust_hid_map,   on='CUSTOMER_NO',              how='left')
    .merge(items_all_cols, on='ITEM_NO',                  how='left', suffixes=('', '_item'))
    .merge(cost_amount,    on=['DOCUMENT_NO', 'LINE_NO'],  how='left')
)

print(f"Order-Zeilen gesamt:        {len(orders_enriched_hid):,}")
print(f"Davon ohne H_ID (kein Mail): {orders_enriched_hid['H_ID'].isna().sum():,}")
print(f"Spalten gesamt:              {len(orders_enriched_hid.columns)}")
display(orders_enriched_hid.head())


Order-Zeilen gesamt:        1,964,146
Davon ohne H_ID (kein Mail): 799
Spalten gesamt:              37


,ITEM_NO,DOCUMENT_NO,LINE_NO,ORDER_DATE,ORDER_NO,POSTING_DATE,COUNTRY_CODE,CUSTOMER_NO,QUANTITY,NET_AMOUNT,...,SUB_CATEGORY,PRODUCT_TYPE,MATERIAL,SIZE,CATEGORY_KEY,PLANNING_CATEGORY_KEY,APPEARANCE,APPEARANCE_EN,CURRENT_PRODUCT_STATUS,COST_AMOUNT
0,AFZ-BNM-001-003,RG2021-21140072,50000.0,2020-11-26,AU2021-21152556,2020-11-28,DE,D535561,2,11.03,...,HEADWEAR,BEANIE,ORGANIC COTTON KNIT,M,AFFENZAHN~APPAREL ACC~HEADWEAR~BEANIE,AFFENZAHN~APPAREL ACC~HEADWEAR~BEANIE~ORGANIC ...,BEAR,NaN,NaN,5.59
1,AFZ-WAL-001-003,GU1920-20004189,40000.0,2019-09-25,AU1920-20081224,2019-10-22,AT,D365138,-1,-12.42,...,WALLETS,WALLET,POLYESTER,onesize,AFFENZAHN~BAGS ACC~WALLETS~WALLET,AFFENZAHN~BAGS ACC~WALLETS~WALLET~~,Bär,Bear,old - older than previous season,-2.29
2,AFZ-SHS-130-224,RG2122-22242797,10000.0,2022-02-28,AU2122-22373259,2022-02-28,DE,D1009168,1,47.06,...,LOWBOOTS,HAPPY,POLYESTER KNIT,30,AFFENZAHN~FOOTWEAR~LOWBOOTS~HAPPY,AFFENZAHN~FOOTWEAR~LOWBOOTS~HAPPY~~,OWL,NaN,old - older than previous season,20.94
3,AFZ-BHE-001-027-4,GU2122-22059117,50000.0,2022-07-11,AU2122-22662834,2022-07-26,DE,D1110671,-1,-41.94,...,EQUIPMENT,HELMET,NaN,M,AFFENZAHN~MOVE ACC~EQUIPMENT~HELMET,AFFENZAHN~MOVE ACC~EQUIPMENT~HELMET~~,UNICORN,NaN,old - older than previous season,-21.45
4,AFZ-FDJ-127-410,RG2122-22334816,10000.0,2022-05-13,AU2122-22532021,2022-05-16,DE,D1051946,1,54.54,...,SANDALS,FREE,POLYESTER CHAMUDE,27,AFFENZAHN~FOOTWEAR~SANDALS~FREE,AFFENZAHN~FOOTWEAR~SANDALS~FREE~~,Vogel,Bird,old - older than previous season,16.14


In [6]:
#Aggregation auf Order-Ebene pro H_ID
order_level_hid = (
    orders_enriched_hid
    .groupby(['H_ID', 'ORDER_NO', 'ORDER_DATE'])
    .agg(
        net_amount_order  = ('NET_AMOUNT',           'sum'),
        cost_amount_order = ('COST_AMOUNT',          'sum'),
        qty_order         = ('QUANTITY',             'sum'),
        n_line_items      = ('ITEM_NO',              'count'),
        n_unique_items    = ('ITEM_NO',              'nunique'),
        n_customer_nos    = ('CUSTOMER_NO',          'nunique'),
        has_discount      = ('LINE_DISCOUNT_AMOUNT', lambda x: (x > 0).any()),
        discount_code     = ('DISCOUNT_CODE',        'first'),
        country_code      = ('COUNTRY_CODE',         'first'),
        posting_date      = ('POSTING_DATE',         'first'),
        season_order_date = ('SEASON_ORDER_DATE',    'first'),
        season_post_date  = ('SEASON_POSTING_DATE',  'first'),
        uvp_sum           = ('UVP',                  'sum'),
        gross_amount      = ('SALES_PRICE_ORIGINAL', 'sum'),
    )
    .reset_index()
)
order_level_hid['is_return_order'] = order_level_hid['qty_order'] <= 0

print(f"Bestellungen gesamt:   {len(order_level_hid):,}")
print(f"Retouren:              {order_level_hid['is_return_order'].sum():,}")
print(f"H_IDs mit >1 CUST_NO: {(order_level_hid['n_customer_nos'] > 1).sum():,}")
display(order_level_hid.head())




Bestellungen gesamt:   966,475
Retouren:              127,394
H_IDs mit >1 CUST_NO: 0


,H_ID,ORDER_NO,ORDER_DATE,net_amount_order,cost_amount_order,qty_order,n_line_items,n_unique_items,n_customer_nos,has_discount,discount_code,country_code,posting_date,season_order_date,season_post_date,uvp_sum,gross_amount,is_return_order
0,000005fbde326c41e4e704c8b358bf34e8edfea6,AU2526-00379151,2026-02-10,54.61,13.26,1,1,1,1,False,None,DE,2026-02-11,SS_2026,SS_2026,64.99,54.61,False
1,00001914a16aa998aa6745e3679d805d6ed05841,AU1920-20321024,2020-06-08,12.52,3.96,1,1,1,1,False,None,DE,2020-06-10,SS_2020,SS_2020,29.90,25.13,False
2,0000342865fa984bb87af10f863e85d6d403e6f6,AU2223-00285821,2023-02-15,63.02,16.49,1,9,5,1,False,None,DE,2023-02-16,SS_2023,SS_2023,634.91,533.54,False
3,0000342865fa984bb87af10f863e85d6d403e6f6,AU2324-00163627,2023-10-26,54.61,18.18,1,9,5,1,False,None,DE,2023-10-27,SS_2023,SS_2023,544.91,457.91,False
4,0000342865fa984bb87af10f863e85d6d403e6f6,AU2324-00387954,2024-04-06,117.63,28.89,2,2,2,1,False,None,DE,2024-04-08,SS_2024,SS_2024,139.98,117.63,False


In [7]:
#Panel: H_ID × Order mit vollem Kundenprofil
panel_hid = (
    order_level_hid
    .merge(customers_hid_profile, on='H_ID', how='left')
    .sort_values(['H_ID', 'ORDER_DATE'])
    .reset_index(drop=True)
)

panel_hid['order_rank'] = (
    panel_hid.groupby('H_ID')['ORDER_DATE']
    .rank(method='dense').astype(int)
)

print(f"Panel (H_ID-basiert): {panel_hid['H_ID'].nunique():,} Kunden | {len(panel_hid):,} Bestellungen")
print(f"Davon Retouren:       {panel_hid['is_return_order'].sum():,}")
print(f"Spalten gesamt:       {len(panel_hid.columns)}")
display(panel_hid.head(10))


Panel (H_ID-basiert): 534,795 Kunden | 966,475 Bestellungen
Davon Retouren:       127,394
Spalten gesamt:       58


,H_ID,ORDER_NO,ORDER_DATE,net_amount_order,cost_amount_order,qty_order,n_line_items,n_unique_items,n_customer_nos,has_discount,...,FIRST_ORDER_NET_AMOUNT_AFTER_RETURNS,FIRST_ORDER_GROSS_AMOUNT_BEFORE_RETURNS,FIRST_ORDER_GROSS_AMOUNT_AFTER_RETURNS,IS_SUBSCRIBED_FOR,IS_OS_B2B_CUSTOMER,DAYS_SINCE_FIRST_ORDER,DAYS_SINCE_LAST_ORDER,CUSTOMER_CREATED_AT,HAS_CUSTOMER_ACCOUNT,order_rank
0,000005fbde326c41e4e704c8b358bf34e8edfea6,AU2526-00379151,2026-02-10,54.61,13.26,1,1,1,1,False,...,54.61,64.99,64.99,NaN,False,63,63,2026-02-10,False,1
1,00001914a16aa998aa6745e3679d805d6ed05841,AU1920-20321024,2020-06-08,12.52,3.96,1,1,1,1,False,...,12.52,14.90,14.90,NaN,False,2137,2137,2020-06-08,False,1
2,0000342865fa984bb87af10f863e85d6d403e6f6,AU2223-00285821,2023-02-15,63.02,16.49,1,9,5,1,False,...,63.02,354.95,74.99,AFFENZAHN,False,1155,46,2023-02-15,True,1
3,0000342865fa984bb87af10f863e85d6d403e6f6,AU2324-00163627,2023-10-26,54.61,18.18,1,9,5,1,False,...,63.02,354.95,74.99,AFFENZAHN,False,1155,46,2023-02-15,True,2
4,0000342865fa984bb87af10f863e85d6d403e6f6,AU2324-00387954,2024-04-06,117.63,28.89,2,2,2,1,False,...,63.02,354.95,74.99,AFFENZAHN,False,1155,46,2023-02-15,True,3
5,0000342865fa984bb87af10f863e85d6d403e6f6,AU2324-00543902,2024-07-02,26.46,10.79,1,3,2,1,False,...,63.02,354.95,74.99,AFFENZAHN,False,1155,46,2023-02-15,True,4
6,0000342865fa984bb87af10f863e85d6d403e6f6,AU2425-00130707,2024-09-22,151.24,37.35,2,2,2,1,False,...,63.02,354.95,74.99,AFFENZAHN,False,1155,46,2023-02-15,True,5
7,0000342865fa984bb87af10f863e85d6d403e6f6,AU2425-00315608,2025-01-28,0.00,0.00,0,2,1,1,False,...,63.02,354.95,74.99,AFFENZAHN,False,1155,46,2023-02-15,True,6
8,0000342865fa984bb87af10f863e85d6d403e6f6,AU2425-00373890,2025-03-08,58.81,13.42,1,15,8,1,False,...,63.02,354.95,74.99,AFFENZAHN,False,1155,46,2023-02-15,True,7
9,0000342865fa984bb87af10f863e85d6d403e6f6,AU2425-00456542,2025-04-22,63.02,14.33,1,1,1,1,False,...,63.02,354.95,74.99,AFFENZAHN,False,1155,46,2023-02-15,True,8


In [8]:
# Mapping H_ID → kurze numerische ID (1, 2, 3, ...)
hid_to_int = {hid: i for i, hid in enumerate(panel_hid['H_ID'].dropna().unique(), start=1)}

panel_hid.insert(
    panel_hid.columns.get_loc('H_ID') + 1,  # direkt nach H_ID einfügen
    'CUSTOMER_ID',
    panel_hid['H_ID'].map(hid_to_int)
)

print(f"Unique CUSTOMER_IDs: {panel_hid['CUSTOMER_ID'].nunique():,}")
print(f"Range: {panel_hid['CUSTOMER_ID'].min()} – {panel_hid['CUSTOMER_ID'].max()}")
display(panel_hid[['CUSTOMER_ID', 'H_ID', 'ORDER_NO', 'ORDER_DATE']].head(10))


Unique CUSTOMER_IDs: 534,795
Range: 1 – 534795


,CUSTOMER_ID,H_ID,ORDER_NO,ORDER_DATE
0,1,000005fbde326c41e4e704c8b358bf34e8edfea6,AU2526-00379151,2026-02-10
1,2,00001914a16aa998aa6745e3679d805d6ed05841,AU1920-20321024,2020-06-08
2,3,0000342865fa984bb87af10f863e85d6d403e6f6,AU2223-00285821,2023-02-15
3,3,0000342865fa984bb87af10f863e85d6d403e6f6,AU2324-00163627,2023-10-26
4,3,0000342865fa984bb87af10f863e85d6d403e6f6,AU2324-00387954,2024-04-06
5,3,0000342865fa984bb87af10f863e85d6d403e6f6,AU2324-00543902,2024-07-02
6,3,0000342865fa984bb87af10f863e85d6d403e6f6,AU2425-00130707,2024-09-22
7,3,0000342865fa984bb87af10f863e85d6d403e6f6,AU2425-00315608,2025-01-28
8,3,0000342865fa984bb87af10f863e85d6d403e6f6,AU2425-00373890,2025-03-08
9,3,0000342865fa984bb87af10f863e85d6d403e6f6,AU2425-00456542,2025-04-22


In [11]:
# EMAIL_HASH = langer Hash, H_ID = kurze Zahl ab 1
panel_hid = panel_hid.rename(columns={'H_ID': 'EMAIL_HASH'})

hid_to_int = {
    hid: i
    for i, hid in enumerate(
        panel_hid['EMAIL_HASH'].dropna().unique(), start=1
    )
}

panel_hid.insert(
    panel_hid.columns.get_loc('EMAIL_HASH') + 1,
    'H_ID',
    panel_hid['EMAIL_HASH'].map(hid_to_int).astype('Int64')
)

print(f"Unique H_IDs: {panel_hid['H_ID'].nunique():,}")
print(f"Range: {panel_hid['H_ID'].min()} – {panel_hid['H_ID'].max()}")
display(
    panel_hid[['H_ID', 'EMAIL_HASH', 'CUSTOMER_NO', 'ORDER_NO', 'ORDER_DATE', 'order_rank']]
    .head(10)
)


Unique H_IDs: 534,795
Range: 1 – 534795


,H_ID,EMAIL_HASH,CUSTOMER_NO,ORDER_NO,ORDER_DATE,order_rank
0,1,000005fbde326c41e4e704c8b358bf34e8edfea6,D2362159,AU2526-00379151,2026-02-10,1
1,2,00001914a16aa998aa6745e3679d805d6ed05841,D503510,AU1920-20321024,2020-06-08,1
2,3,0000342865fa984bb87af10f863e85d6d403e6f6,D1301238,AU2223-00285821,2023-02-15,1
3,3,0000342865fa984bb87af10f863e85d6d403e6f6,D1301238,AU2324-00163627,2023-10-26,2
4,3,0000342865fa984bb87af10f863e85d6d403e6f6,D1301238,AU2324-00387954,2024-04-06,3
5,3,0000342865fa984bb87af10f863e85d6d403e6f6,D1301238,AU2324-00543902,2024-07-02,4
6,3,0000342865fa984bb87af10f863e85d6d403e6f6,D1301238,AU2425-00130707,2024-09-22,5
7,3,0000342865fa984bb87af10f863e85d6d403e6f6,D1301238,AU2425-00315608,2025-01-28,6
8,3,0000342865fa984bb87af10f863e85d6d403e6f6,D1301238,AU2425-00373890,2025-03-08,7
9,3,0000342865fa984bb87af10f863e85d6d403e6f6,D1301238,AU2425-00456542,2025-04-22,8


In [14]:
# Panel neu aufbauen + H_ID (kurze Zahl) + EMAIL_HASH (ganz rechts)
panel_hid = (
    order_level_hid
    .merge(customers_hid_profile, on='H_ID', how='left')
    .sort_values(['H_ID', 'ORDER_DATE'])
    .reset_index(drop=True)
)

panel_hid['order_rank'] = (
    panel_hid.groupby('H_ID')['ORDER_DATE']
    .rank(method='dense').astype(int)
)

# Hash umbenennen
panel_hid = panel_hid.rename(columns={'H_ID': 'EMAIL_HASH'})

# Kurze numerische H_ID ab 1
hid_to_int = {
    hid: i
    for i, hid in enumerate(panel_hid['EMAIL_HASH'].dropna().unique(), start=1)
}
panel_hid.insert(0, 'H_ID', panel_hid['EMAIL_HASH'].map(hid_to_int).astype('Int64'))

# EMAIL_HASH ganz nach rechts
panel_hid = panel_hid[[c for c in panel_hid.columns if c != 'EMAIL_HASH'] + ['EMAIL_HASH']]

print(f"Panel (H_ID-basiert): {panel_hid['H_ID'].nunique():,} Kunden | {len(panel_hid):,} Bestellungen")
print(f"Davon Retouren:       {panel_hid['is_return_order'].sum():,}")
print(f"Spalten gesamt:       {len(panel_hid.columns)}")
display(
    panel_hid[['H_ID', 'CUSTOMER_NO', 'ORDER_NO', 'ORDER_DATE', 'order_rank', 'EMAIL_HASH']]
    .head(10)
)


Panel (H_ID-basiert): 534,795 Kunden | 966,475 Bestellungen
Davon Retouren:       127,394
Spalten gesamt:       59


,H_ID,CUSTOMER_NO,ORDER_NO,ORDER_DATE,order_rank,EMAIL_HASH
0,1,D2362159,AU2526-00379151,2026-02-10,1,000005fbde326c41e4e704c8b358bf34e8edfea6
1,2,D503510,AU1920-20321024,2020-06-08,1,00001914a16aa998aa6745e3679d805d6ed05841
2,3,D1301238,AU2223-00285821,2023-02-15,1,0000342865fa984bb87af10f863e85d6d403e6f6
3,3,D1301238,AU2324-00163627,2023-10-26,2,0000342865fa984bb87af10f863e85d6d403e6f6
4,3,D1301238,AU2324-00387954,2024-04-06,3,0000342865fa984bb87af10f863e85d6d403e6f6
5,3,D1301238,AU2324-00543902,2024-07-02,4,0000342865fa984bb87af10f863e85d6d403e6f6
6,3,D1301238,AU2425-00130707,2024-09-22,5,0000342865fa984bb87af10f863e85d6d403e6f6
7,3,D1301238,AU2425-00315608,2025-01-28,6,0000342865fa984bb87af10f863e85d6d403e6f6
8,3,D1301238,AU2425-00373890,2025-03-08,7,0000342865fa984bb87af10f863e85d6d403e6f6
9,3,D1301238,AU2425-00456542,2025-04-22,8,0000342865fa984bb87af10f863e85d6d403e6f6


In [15]:
panel_hid.to_csv('Processed Data/big_panel.csv', index=False)

print(f"Gespeichert: 'Processed Data/big_panel.csv'")
print(f"Zeilen: {len(panel_hid):,} | Spalten: {len(panel_hid.columns)}")


Gespeichert: 'Processed Data/big_panel.csv'
Zeilen: 966,475 | Spalten: 59


Merge PLZ Income and Kids statistics

In [ ]:
kids_raw      = pd.read_csv('Data/kids_per_bundesland.csv')
#https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Bevoelkerung/Haushalte-Familien/Tabellen/2-7-familien-bundeslaender.html
income_raw    = pd.read_csv('Data/income_per_city_region_2023.csv')
#https://www.statistikportal.de/de/vgrdl/ergebnisse-kreisebene/einkommen-kreise
#Einkommen der priaten Haushalte
plz_mapping   = pd.read_csv('Data/mapping_postcode.csv')
#https://gist.github.com/pmdroid/6ae8286a494cafce82b6ea5f6cc2362a#file-german-postcodes-csv

In [29]:
#Quelle
kids_raw = pd.read_csv('Data/kids_per_bundesland.csv')

aggregat_zeilen = ['Deutschland', 'Westdeutschland ohne Berlin', 'Ostdeutschland einschließlich Berlin']
kids = kids_raw[~kids_raw['Bundesland'].isin(aggregat_zeilen)].reset_index(drop=True)

print(f"Bundesländer: {len(kids)}")
display(kids)


Bundesländer: 16


,Bundesland,Familien_gesamt_1000,Kinder_1_1000,Kinder_2_1000,Kinder_3plus_1000,Familienmitglieder_gesamt_1000,Familienmitglieder_je_Familie
0,Baden-Württemberg,1136,527,456,154,4259,3.75
1,Bayern,1355,651,550,155,4993,3.68
2,Bremen,69,34,24,11,254,3.71
3,Hamburg,196,98,75,23,707,3.61
4,Hessen,642,314,244,84,2360,3.68
5,Niedersachsen,788,371,309,108,2926,3.71
6,Nordrhein-Westfalen,1791,854,692,245,6644,3.71
7,Rheinland-Pfalz,411,201,158,53,1516,3.68
8,Saarland,94,48,34,13,348,3.70
9,Schleswig-Holstein,290,144,111,36,1050,3.62


In [22]:
income_raw = pd.read_csv('Data/income_per_city_region_2023.csv')

print(f"Zeilen: {len(income_raw):,} | nuts_level-Verteilung:")
print(income_raw['nuts_level'].value_counts().sort_index())
display(income_raw.head())


Zeilen: 444 | nuts_level-Verteilung:
nuts_level
1     17
2     29
3    398
Name: count, dtype: int64


,lfd_nr,eu_code,regionalschluessel,land,nuts_level,gebietseinheit,y2018,y2019,y2020,y2021,y2022,y2023
0,1,DE1,8,BW,1,Baden-Württemberg,25809,26164,25842,26626,28520,30242
1,2,DE11,81,BW,2,"Stuttgart, Regierungsbezirk",26355,26753,26206,27248,29219,31022
2,3,DE111,8111,BW,3,"Stuttgart, Landeshauptstadt, Stadtkreis",26985,27066,26673,27283,29413,31390
3,4,DE112,8115,BW,3,"Böblingen, Landkreis",26694,27242,26967,27540,29208,31100
4,5,DE113,8116,BW,3,"Esslingen, Landkreis",26782,26944,26421,27375,29170,31076


In [24]:
income_city = (
    income_raw[income_raw['nuts_level'] == 3]
    [['gebietseinheit', 'land', 'regionalschluessel', 'y2023']]
    .copy()
    .rename(columns={'y2023': 'kaufkraft_2023'})
)

# Kernname: alles vor erstem Komma, Suffixe entfernen, lowercase
income_city['city_clean'] = (
    income_city['gebietseinheit']
    .str.split(',').str[0]
    .str.strip()
    .str.replace(r'\s+(Landkreis|Stadtkreis|Kreis|Stadt|Regierungsbezirk)$', '', regex=True)
    .str.strip()
    .str.lower()
)

# Bei Duplikaten (z.B. Karlsruhe Stadtkreis + Landkreis): Stadtkreis bevorzugen
income_city['_stadtkreis'] = income_city['gebietseinheit'].str.contains('Stadtkreis', case=False)
income_city = (
    income_city
    .sort_values('_stadtkreis', ascending=False)
    .drop_duplicates(subset='city_clean')
    .drop(columns='_stadtkreis')
    .reset_index(drop=True)
)

print(f"Income-Einträge (nuts=3, dedup): {len(income_city):,}")
display(income_city[['gebietseinheit', 'city_clean', 'kaufkraft_2023']].head(10))


Income-Einträge (nuts=3, dedup): 377


,gebietseinheit,city_clean,kaufkraft_2023
0,"Stuttgart, Landeshauptstadt, Stadtkreis",stuttgart,31390
1,"Freiburg im Breisgau, Stadtkreis",freiburg im breisgau,27158
2,"Karlsruhe, Stadtkreis",karlsruhe,28467
3,"Baden-Baden, Stadtkreis",baden-baden,35144
4,"Pforzheim, Stadtkreis",pforzheim,26823
5,"Heidelberg, Stadtkreis",heidelberg,30764
6,"Ulm, Universitätsstadt, Stadtkreis",ulm,32974
7,"Heilbronn, Stadtkreis",heilbronn,40646
8,"Mannheim, Universitätsstadt, Stadtkreis",mannheim,26853
9,"Recklinghausen, Kreis",recklinghausen,25686


In [25]:
customers_city = (
    customers_all_hid[customers_all_hid['COUNTRY_REGION_CODE'] == 'DE']
    [['POST_CODE', 'CITY']].drop_duplicates().dropna(subset=['CITY'])
)
customers_city['POST_CODE'] = customers_city['POST_CODE'].astype(str).str.zfill(5)

customers_city['city_clean'] = (
    customers_city['CITY']
    .str.strip()
    .str.replace(r'\s*/\s*', ' ', regex=True)   # "Halle / Saale" → "Halle  Saale"
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)

print(f"Unique PLZ/Ort-Kombinationen: {len(customers_city):,}")
display(customers_city.head(10))


Unique PLZ/Ort-Kombinationen: 30,421


,POST_CODE,CITY,city_clean
0,30926,Seelze,seelze
2,56179,Vallendar,vallendar
4,06729,Elsteraue,elsteraue
5,50667,Köln,köln
6,72149,Neustetten,neustetten
7,10179,Berlin,berlin
8,63110,Rodgau,rodgau
9,20095,Hamburg,hamburg
10,38372,Büddenstedt,büddenstedt
12,73492,Rainau,rainau


In [30]:
plz_mapping = pd.read_csv('Data/mapping_postcode.csv')
plz_mapping['Plz'] = plz_mapping['Plz'].astype(str).str.zfill(5)

print(f"Einträge: {len(plz_mapping):,} | Unique PLZs: {plz_mapping['Plz'].nunique():,}")
display(plz_mapping.head())


Einträge: 19,671 | Unique PLZs: 8,256


,Ort,Plz,Bundesland
0,Aach,54298,Rheinland-Pfalz
1,Aach,78267,Baden-Württemberg
2,Aachen,52062,Nordrhein-Westfalen
3,Aachen,52064,Nordrhein-Westfalen
4,Aachen,52066,Nordrhein-Westfalen


In [31]:
merged_bl = (
    customers_city
    .merge(
        plz_mapping[['Plz', 'Bundesland']].drop_duplicates(subset='Plz'),
        left_on='POST_CODE', right_on='Plz',
        how='left'
    )
    .drop(columns='Plz')
    .merge(kids, on='Bundesland', how='left')
)

print(f"Zeilen:                {len(merged_bl):,}")
print(f"Ohne Bundesland-Match: {merged_bl['Bundesland'].isna().sum():,}")
display(merged_bl.head(10))


Zeilen:                30,421
Ohne Bundesland-Match: 1,865


,POST_CODE,CITY,city_clean,Bundesland,Familien_gesamt_1000,Kinder_1_1000,Kinder_2_1000,Kinder_3plus_1000,Familienmitglieder_gesamt_1000,Familienmitglieder_je_Familie
0,30926,Seelze,seelze,Niedersachsen,788.0,371.0,309.0,108.0,2926.0,3.71
1,56179,Vallendar,vallendar,Rheinland-Pfalz,411.0,201.0,158.0,53.0,1516.0,3.68
2,06729,Elsteraue,elsteraue,Sachsen-Anhalt,199.0,112.0,65.0,22.0,689.0,3.46
3,50667,Köln,köln,Nordrhein-Westfalen,1791.0,854.0,692.0,245.0,6644.0,3.71
4,72149,Neustetten,neustetten,Baden-Württemberg,1136.0,527.0,456.0,154.0,4259.0,3.75
5,10179,Berlin,berlin,Berlin,381.0,196.0,140.0,45.0,1352.0,3.55
6,63110,Rodgau,rodgau,Hessen,642.0,314.0,244.0,84.0,2360.0,3.68
7,20095,Hamburg,hamburg,Hamburg,196.0,98.0,75.0,23.0,707.0,3.61
8,38372,Büddenstedt,büddenstedt,Niedersachsen,788.0,371.0,309.0,108.0,2926.0,3.71
9,73492,Rainau,rainau,Baden-Württemberg,1136.0,527.0,456.0,154.0,4259.0,3.75


In [32]:
# Cleaning: Ort aus plz_mapping & CITY aus customers
merged_bl['city_clean'] = merged_bl['CITY'].str.strip().str.lower()

income_city['city_clean'] = (
    income_raw[income_raw['nuts_level'] == 3]['gebietseinheit']
    .str.split(',').str[0]
    .str.replace(r'\s+(Landkreis|Stadtkreis|Kreis|Stadt)$', '', regex=True)
    .str.strip()
    .str.lower()
)

# Check: welche Kundenstädte matchen NICHT?
unmatched_cities = set(merged_bl['city_clean']) - set(income_city['city_clean'])
print(f"Städte ohne Match: {len(unmatched_cities):,} von {merged_bl['city_clean'].nunique():,}")
print("Beispiele:", sorted(unmatched_cities)[:20])


Städte ohne Match: 20,790 von 20,997
Beispiele: ["'schonach", '(alle)', ', greifswald', '----------geislingen', '.cottbus', '01159  dresden', '01445 radebeul', '016092021091', '04229 leipzig', '04299 leipzig', '06449 aschersleben', '06502 thale', '06712', '09648 altmittweida', '0bertshausen', '10179', '12169 berlin', '13409 berlin', '13465 berlin', '13469 berlin']


In [33]:
income_clean = (
    income_raw[income_raw['nuts_level'] == 3]
    [['gebietseinheit', 'y2023']]
    .copy()
    .rename(columns={'y2023': 'kaufkraft_2023'})
)
income_clean['city_clean'] = (
    income_clean['gebietseinheit']
    .str.split(',').str[0]
    .str.replace(r'\s+(Landkreis|Stadtkreis|Kreis|Stadt)$', '', regex=True)
    .str.strip()
    .str.lower()
)
income_clean = income_clean.drop_duplicates(subset='city_clean')

plz_stats = merged_bl.merge(income_clean, on='city_clean', how='left')

matched   = plz_stats['kaufkraft_2023'].notna().sum()
unmatched = plz_stats['kaufkraft_2023'].isna().sum()
print(f"Gesamt:        {len(plz_stats):,}")
print(f"Mit Ort-Match: {matched:,}  ({matched/len(plz_stats)*100:.1f}%)")
print(f"Ohne Match:    {unmatched:,}")
display(plz_stats.head(10))


Gesamt:        30,421
Mit Ort-Match: 3,024  (9.9%)
Ohne Match:    27,397


,POST_CODE,CITY,city_clean,Bundesland,Familien_gesamt_1000,Kinder_1_1000,Kinder_2_1000,Kinder_3plus_1000,Familienmitglieder_gesamt_1000,Familienmitglieder_je_Familie,gebietseinheit,kaufkraft_2023
0,30926,Seelze,seelze,Niedersachsen,788.0,371.0,309.0,108.0,2926.0,3.71,NaN,NaN
1,56179,Vallendar,vallendar,Rheinland-Pfalz,411.0,201.0,158.0,53.0,1516.0,3.68,NaN,NaN
2,06729,Elsteraue,elsteraue,Sachsen-Anhalt,199.0,112.0,65.0,22.0,689.0,3.46,NaN,NaN
3,50667,Köln,köln,Nordrhein-Westfalen,1791.0,854.0,692.0,245.0,6644.0,3.71,"Köln, Kreisfreie Stadt",28764.0
4,72149,Neustetten,neustetten,Baden-Württemberg,1136.0,527.0,456.0,154.0,4259.0,3.75,NaN,NaN
5,10179,Berlin,berlin,Berlin,381.0,196.0,140.0,45.0,1352.0,3.55,NaN,NaN
6,63110,Rodgau,rodgau,Hessen,642.0,314.0,244.0,84.0,2360.0,3.68,NaN,NaN
7,20095,Hamburg,hamburg,Hamburg,196.0,98.0,75.0,23.0,707.0,3.61,NaN,NaN
8,38372,Büddenstedt,büddenstedt,Niedersachsen,788.0,371.0,309.0,108.0,2926.0,3.71,NaN,NaN
9,73492,Rainau,rainau,Baden-Württemberg,1136.0,527.0,456.0,154.0,4259.0,3.75,NaN,NaN


second try 


In [34]:
# mapping_postcode: saubere Ort-Namen pro PLZ
plz_mapping['Plz'] = plz_mapping['Plz'].astype(str).str.zfill(5)
plz_mapping['city_clean'] = plz_mapping['Ort'].str.strip().str.lower()

# income: nur nuts_level=3 (Landkreis-Ebene, granularst verfügbar)
income_nuts3 = (
    income_raw[income_raw['nuts_level'] == 3]
    [['gebietseinheit', 'regionalschluessel', 'land', 'y2023']]
    .copy()
    .rename(columns={'y2023': 'kaufkraft_2023'})
)
income_nuts3['city_clean'] = (
    income_nuts3['gebietseinheit']
    .str.split(',').str[0]
    .str.strip()
    .str.lower()
)
# Bei Duplikaten: Stadtkreis bevorzugen (präziser als Landkreis)
income_nuts3['_prio'] = income_nuts3['gebietseinheit'].str.contains(
    'Stadtkreis|Universitätsstadt|Landeshauptstadt|Freie.*Stadt', case=False, regex=True
)
income_nuts3 = (
    income_nuts3.sort_values('_prio', ascending=False)
    .drop_duplicates(subset='city_clean')
    .drop(columns='_prio')
    .reset_index(drop=True)
)

print(f"income-Einträge (nuts=3, dedup): {len(income_nuts3):,}")
display(income_nuts3[['gebietseinheit', 'city_clean', 'kaufkraft_2023']].head(10))


income-Einträge (nuts=3, dedup): 377


,gebietseinheit,city_clean,kaufkraft_2023
0,"Stuttgart, Landeshauptstadt, Stadtkreis",stuttgart,31390
1,"Ludwigshafen am Rhein, Kreisfreie Stadt",ludwigshafen am rhein,23283
2,"Schweinfurt, Kreisfreie Stadt",schweinfurt,26074
3,"Würzburg, Kreisfreie Stadt",würzburg,29291
4,"Landau in der Pfalz, Kreisfreie Stadt",landau in der pfalz,26213
5,"Kaiserslautern, Kreisfreie Stadt",kaiserslautern,22631
6,"Frankenthal (Pfalz), Kreisfreie Stadt",frankenthal (pfalz),25477
7,"Trier, Kreisfreie Stadt",trier,26127
8,"Augsburg, Kreisfreie Stadt",augsburg,25528
9,"Kaufbeuren, Kreisfreie Stadt",kaufbeuren,28049


In [35]:
# Unique PLZs aus Kundendaten (nur DE)
unique_plzs = (
    customers_all_hid[customers_all_hid['COUNTRY_REGION_CODE'] == 'DE']
    [['POST_CODE']].drop_duplicates().dropna()
)
unique_plzs['POST_CODE'] = unique_plzs['POST_CODE'].astype(str).str.zfill(5)

# PLZ → sauberer Ort + Bundesland (aus mapping_postcode, nicht aus schmutzigem CITY!)
plz_enriched = unique_plzs.merge(
    plz_mapping[['Plz', 'Ort', 'Bundesland', 'city_clean']].drop_duplicates(subset='Plz'),
    left_on='POST_CODE', right_on='Plz', how='left'
).drop(columns='Plz')

# Left join kids via Bundesland
plz_enriched = plz_enriched.merge(kids, on='Bundesland', how='left')

print(f"Unique PLZs:           {len(plz_enriched):,}")
print(f"Ohne Bundesland-Match: {plz_enriched['Bundesland'].isna().sum():,}")
display(plz_enriched.head(10))


Unique PLZs:           9,871
Ohne Bundesland-Match: 1,757


,POST_CODE,Ort,Bundesland,city_clean,Familien_gesamt_1000,Kinder_1_1000,Kinder_2_1000,Kinder_3plus_1000,Familienmitglieder_gesamt_1000,Familienmitglieder_je_Familie
0,30926,Seelze,Niedersachsen,seelze,788.0,371.0,309.0,108.0,2926.0,3.71
1,56179,Niederwerth,Rheinland-Pfalz,niederwerth,411.0,201.0,158.0,53.0,1516.0,3.68
2,06729,Etzoldshain,Sachsen-Anhalt,etzoldshain,199.0,112.0,65.0,22.0,689.0,3.46
3,50667,Köln,Nordrhein-Westfalen,köln,1791.0,854.0,692.0,245.0,6644.0,3.71
4,72149,Neustetten,Baden-Württemberg,neustetten,1136.0,527.0,456.0,154.0,4259.0,3.75
5,10179,Berlin,Berlin,berlin,381.0,196.0,140.0,45.0,1352.0,3.55
6,63110,Rodgau,Hessen,rodgau,642.0,314.0,244.0,84.0,2360.0,3.68
7,20095,Hamburg,Hamburg,hamburg,196.0,98.0,75.0,23.0,707.0,3.61
8,38372,Büddenstedt,Niedersachsen,büddenstedt,788.0,371.0,309.0,108.0,2926.0,3.71
9,73492,Rainau,Baden-Württemberg,rainau,1136.0,527.0,456.0,154.0,4259.0,3.75


In [36]:
plz_stats = plz_enriched.merge(
    income_nuts3[['city_clean', 'gebietseinheit', 'regionalschluessel', 'kaufkraft_2023']],
    on='city_clean', how='left'
)

matched   = plz_stats['kaufkraft_2023'].notna().sum()
unmatched = plz_stats['kaufkraft_2023'].isna().sum()
print(f"Gesamt:                {len(plz_stats):,}")
print(f"Mit Landkreis-Match:   {matched:,}  ({matched/len(plz_stats)*100:.1f}%)")
print(f"Ohne Match (nur BL):   {unmatched:,}")

# Ungematchte Orte zeigen
print("\nTop ungematchte Orte:")
print(plz_stats[plz_stats['kaufkraft_2023'].isna()]['Ort'].value_counts().head(15))
display(plz_stats.head(10))


Gesamt:                9,871
Mit Landkreis-Match:   1,197  (12.1%)
Ohne Match (nur BL):   8,674

Top ungematchte Orte:
Ort
Berlin        190
Hamburg        98
Frankfurt      42
Hannover       27
Halle          14
Freiburg       13
Neustadt       11
Mülheim        10
Aachen         10
Wetzlar        10
Hausen          9
Limburg         8
Dillenburg      8
Neuss           8
Neukirchen      8
Name: count, dtype: int64


,POST_CODE,Ort,Bundesland,city_clean,Familien_gesamt_1000,Kinder_1_1000,Kinder_2_1000,Kinder_3plus_1000,Familienmitglieder_gesamt_1000,Familienmitglieder_je_Familie,gebietseinheit,regionalschluessel,kaufkraft_2023
0,30926,Seelze,Niedersachsen,seelze,788.0,371.0,309.0,108.0,2926.0,3.71,NaN,NaN,NaN
1,56179,Niederwerth,Rheinland-Pfalz,niederwerth,411.0,201.0,158.0,53.0,1516.0,3.68,NaN,NaN,NaN
2,06729,Etzoldshain,Sachsen-Anhalt,etzoldshain,199.0,112.0,65.0,22.0,689.0,3.46,NaN,NaN,NaN
3,50667,Köln,Nordrhein-Westfalen,köln,1791.0,854.0,692.0,245.0,6644.0,3.71,"Köln, Kreisfreie Stadt",5315.0,28764.0
4,72149,Neustetten,Baden-Württemberg,neustetten,1136.0,527.0,456.0,154.0,4259.0,3.75,NaN,NaN,NaN
5,10179,Berlin,Berlin,berlin,381.0,196.0,140.0,45.0,1352.0,3.55,NaN,NaN,NaN
6,63110,Rodgau,Hessen,rodgau,642.0,314.0,244.0,84.0,2360.0,3.68,NaN,NaN,NaN
7,20095,Hamburg,Hamburg,hamburg,196.0,98.0,75.0,23.0,707.0,3.61,NaN,NaN,NaN
8,38372,Büddenstedt,Niedersachsen,büddenstedt,788.0,371.0,309.0,108.0,2926.0,3.71,NaN,NaN,NaN
9,73492,Rainau,Baden-Württemberg,rainau,1136.0,527.0,456.0,154.0,4259.0,3.75,NaN,NaN,NaN
